# 13장 웹부록 | 가상의 뉴런 신호로 커서 움직이기

「영화로 보는 AI 리터러시」 13장 '생각을 읽는 기계: 뇌-컴퓨터 인터페이스'의 코랩 실습입니다.

**쓰는 법**

- 코드를 몰라도 괜찮습니다. 위에서부터 차례로 셀 왼쪽의 ▶(실행) 버튼을 누르세요.
- `# ← 이 숫자를 바꿔 보세요` 표시가 있는 곳만 고치고, 그 셀부터 다시 실행해 보세요.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 따로 설치할 것은 없습니다. 그래프 글자는 영어로 나오고, 설명은 이 글에 우리말로 적었습니다.

**이 실습에서 할 일**

1. 좋아하는 방향이 서로 다른 **가상 뉴런 50개**의 스파이크를 만듭니다.
2. 보정 데이터를 모으고, **집단 벡터**와 간단한 **선형 해독기**로 커서가 갈 방향을 알아맞힙니다.
3. 해독한 방향으로 화면 속 커서를 움직여 봅니다.
4. 보정 데이터의 양을 줄이거나 뉴런 일부를 '끊어' 보면서 정확도가 어떻게 변하는지 그래프로 확인합니다.

책 3-3절(BCI는 이렇게 일합니다), 3-4절(뇌 신호를 해독하는 AI), 실습 1(종이 뇌 해독기 만들기)을 함께 보세요. 이 실습의 뉴런은 모두 컴퓨터가 지어낸 가상 뉴런입니다.

## 1단계. 가상 뉴런 50개 만들기

책에서 본 게오르고풀로스의 발견처럼, 뉴런마다 **좋아하는 방향**이 있습니다. 좋아하는 방향으로 팔(또는 커서)을 움직이려 할 때 스파이크를 가장 많이 쏘고, 반대 방향일 때 가장 적게 쏩니다. 옆 방향이면 그 중간입니다.

다만 진짜 뉴런처럼 **늘 조금씩 들쭉날쭉**합니다. 같은 방향을 생각해도 매번 스파이크 수가 다릅니다. 종이 실습에서 손가락 수를 '일부러 조금씩 흔들리게' 한 것과 같습니다.

👉 **바꿔 볼 곳**: 뉴런 수 `N_NEURONS`를 4로 줄이면 종이 실습과 같은 크기가 됩니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

N_NEURONS = 50   # ← 이 숫자를 바꿔 보세요 (가상 뉴런 수)
WINDOW = 0.1     # 스파이크를 세는 시간 창(초). 0.1초마다 한 번씩 해독합니다

rng = np.random.default_rng(13)
pref = rng.uniform(0, 2*np.pi, N_NEURONS)      # 뉴런마다 좋아하는 방향(진짜 값, 해독기는 모릅니다)
base = rng.uniform(5, 15, N_NEURONS)          # 평소 1초에 쏘는 스파이크 수
depth = rng.uniform(3, 12, N_NEURONS)         # 방향에 따라 늘고 주는 폭

def spikes(direction, rng, alive=None):
    """'direction' 쪽으로 움직이려 할 때 0.1초 동안 뉴런마다 쏜 스파이크 수"""
    direction = np.atleast_1d(direction)
    rate = base + depth * np.cos(direction[:, None] - pref[None, :])   # 1초당 스파이크 수
    counts = rng.poisson(np.maximum(rate, 0) * WINDOW)                  # 들쭉날쭉한 실제 스파이크
    if alive is not None:
        counts = counts * alive                                          # 끊어진 뉴런은 0
    return counts

# 뉴런 4개의 '방향 취향'을 그려 봅니다
angles = np.linspace(0, 2*np.pi, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for i in range(4):
    axes[0].plot(np.degrees(angles), base[i] + depth[i]*np.cos(angles - pref[i]), label=f"neuron {i+1} (likes {np.degrees(pref[i]):.0f} deg)")
axes[0].set_xlabel("movement direction (degrees)"); axes[0].set_ylabel("spikes per second")
axes[0].set_title("Each neuron has a preferred direction"); axes[0].legend(fontsize=8)

# '오른쪽(0도)'을 생각할 때 50개 뉴런이 쏜 스파이크(1초 동안, 0.1초 x 10)
order = np.argsort(pref)
raster = np.vstack([spikes(0.0, rng) for _ in range(10)]).sum(0)
axes[1].bar(np.degrees(pref[order]), raster[order], width=5)
axes[1].set_xlabel("neuron's preferred direction (degrees)"); axes[1].set_ylabel("spikes in 1 s")
axes[1].set_title("Thinking 'right' (0 deg): who fires most?")
plt.tight_layout(); plt.show()

오른쪽 그림에서 막대 하나가 뉴런 하나입니다. '오른쪽(0도)'을 생각하면, 0도나 360도 근처를 좋아하는 뉴런이 가장 많이 쏩니다. 하지만 막대가 들쭉날쭉해서 뉴런 하나만 보고는 방향을 정확히 알기 어렵습니다. 여럿을 함께 봐야 합니다.

## 2단계. 보정: 정답이 있는 데이터 모으기

BCI는 쓰기 전에 **보정**(calibration)을 합니다. 화면의 점이 여덟 방향 중 하나로 움직이면, 사용자는 '나도 저쪽으로 움직인다'고 상상합니다. 이때의 스파이크 수와 정답 방향을 짝지어 모읍니다. 3장의 **지도학습** 데이터와 같습니다.

👉 **바꿔 볼 곳**: 보정 횟수 `N_CALIB`를 바꿔 보세요. 한 번은 0.1초 분량입니다.

In [ ]:
N_CALIB = 200   # ← 이 숫자를 바꿔 보세요 (보정 횟수)

EIGHT = np.arange(8) * np.pi / 4            # 여덟 방향: 0, 45, 90, ... 315도

def collect(n, rng, alive=None):
    dirs = rng.choice(EIGHT, n)
    return dirs, spikes(dirs, rng, alive)

calib_dirs, calib_counts = collect(N_CALIB, rng)
print("보정 데이터 크기:", calib_counts.shape, "(보정 횟수 x 뉴런 수)")
print("첫 보정 3번의 정답 방향(도):", np.degrees(calib_dirs[:3]).round())
print("그때 뉴런 1~8번의 스파이크 수:")
print(calib_counts[:3, :8])

## 3단계. 두 가지 해독기 만들기

**① 집단 벡터 해독기** (책 3-4절)

1. 보정 데이터로 뉴런마다 '좋아하는 방향'과 '평소 스파이크 수'를 알아냅니다. 해독기는 1단계의 진짜 값을 모르고, 보정 데이터만 봅니다.
2. 사용할 때는 뉴런마다 좋아하는 방향으로 화살표를 그리고, 평소보다 많이 쏜 만큼 화살표를 길게 합니다.
3. 모든 화살표를 더한 방향이 답입니다. 뉴런들의 **투표**입니다.

**② 선형 해독기** (책의 '해독기의 속' 상자)

스파이크 수마다 가중치를 곱해 더하면 방향(가로 성분, 세로 성분)이 나오도록, 보정 데이터에 가장 잘 맞는 가중치를 계산으로 찾습니다. 뉴런이 50개면 가중치도 50개 남짓입니다. 보정 데이터가 적을 때 가중치가 엉뚱하게 커지지 않도록 살짝 눌러 주는 장치(`ridge`)를 넣었습니다.

두 해독기 모두 새로 만든 시험 데이터 400번으로 채점합니다. 해독한 방향이 여덟 방향 중 정답과 가장 가까우면 '맞음'입니다.

In [ ]:
def fit_population_vector(dirs, counts):
    """보정 데이터로 뉴런마다 평소 스파이크 수와 좋아하는 방향을 추정합니다."""
    X = np.c_[np.ones(len(dirs)), np.cos(dirs), np.sin(dirs)]
    coef, *_ = np.linalg.lstsq(X, counts, rcond=None)       # 뉴런마다: 평소값, 가로, 세로
    return {"base": coef[0], "pref": np.arctan2(coef[2], coef[1]), "gain": np.hypot(coef[1], coef[2]) + 1e-9}

def decode_population_vector(pv, counts):
    votes = (counts - pv["base"]) / pv["gain"]                # 평소보다 얼마나 더 쐈나
    x = (votes * np.cos(pv["pref"])).sum(-1); y = (votes * np.sin(pv["pref"])).sum(-1)
    return np.arctan2(y, x)

def fit_linear(dirs, counts, ridge=20.0):
    X = np.c_[np.ones(len(counts)), counts]
    Y = np.c_[np.cos(dirs), np.sin(dirs)]
    R = ridge * np.eye(X.shape[1]); R[0, 0] = 0          # 가중치가 너무 커지지 않게 살짝 눌러 줍니다
    return np.linalg.solve(X.T @ X + R, X.T @ Y)

def decode_linear(W, counts):
    out = np.c_[np.ones(len(counts)), counts] @ W
    return np.arctan2(out[:, 1], out[:, 0])

def accuracy(decoded, truth):
    nearest = np.round(decoded / (np.pi/4)) % 8
    return (nearest == np.round(truth / (np.pi/4)) % 8).mean()

def angle_error(decoded, truth):
    return np.degrees(np.abs((decoded - truth + np.pi) % (2*np.pi) - np.pi)).mean()

pv = fit_population_vector(calib_dirs, calib_counts)
W = fit_linear(calib_dirs, calib_counts)

test_dirs, test_counts = collect(400, np.random.default_rng(99))
rows = []
for name, dec in [("집단 벡터", decode_population_vector(pv, test_counts)), ("선형 해독기", decode_linear(W, test_counts))]:
    rows.append({"해독기": name, "여덟 방향 맞힌 비율": f"{accuracy(dec, test_dirs):.0%}",
                 "평균 각도 오차(도)": round(angle_error(dec, test_dirs), 1)})
display(pd.DataFrame(rows))

err = np.degrees(np.abs((pv["pref"] - pref + np.pi) % (2*np.pi) - np.pi))
print(f"집단 벡터 해독기가 보정으로 알아낸 '좋아하는 방향'의 평균 오차: {err.mean():.1f}도")

## 4단계. 커서 움직이기

이제 화면 가운데의 커서를 여덟 개의 표적으로 움직여 봅니다. 사용자는 0.1초마다 '표적 쪽으로!'라고 생각하고, 해독기는 스파이크를 보고 방향을 알아내 커서를 그쪽으로 조금 옮깁니다. 책 3-3절의 '측정 → 해독 → 출력 → 피드백' 순환입니다. 사용자는 커서가 어긋나면 다시 표적 쪽을 생각해 바로잡습니다.

👉 **바꿔 볼 곳**: `DECODER`를 `"linear"`로 바꿔 보세요.

In [ ]:
DECODER = "population"   # ← "population"(집단 벡터) 또는 "linear"(선형 해독기)
STEP = 0.08                # 0.1초마다 커서가 움직이는 거리

def move_cursor(target_angle, decode, rng, alive=None, max_steps=60):
    goal = np.array([np.cos(target_angle), np.sin(target_angle)])
    pos, path = np.zeros(2), [np.zeros(2)]
    for t in range(max_steps):
        intended = np.arctan2(*(goal - pos)[::-1])               # 사용자가 생각하는 방향(피드백)
        counts = spikes(intended, rng, alive)
        d = decode(counts)[0]
        pos = pos + STEP * np.array([np.cos(d), np.sin(d)])
        path.append(pos.copy())
        if np.linalg.norm(goal - pos) < 0.1:
            return np.array(path), t + 1
    return np.array(path), None

decoders = {"population": lambda c: decode_population_vector(pv, c), "linear": lambda c: decode_linear(W, c)}
crng = np.random.default_rng(5)
fig, ax = plt.subplots(figsize=(5, 5))
times = []
for a in EIGHT:
    path, t = move_cursor(a, decoders[DECODER], crng)
    times.append(t)
    ax.plot(path[:, 0], path[:, 1], "-", lw=1.5)
    ax.plot(np.cos(a), np.sin(a), "o", ms=18, mfc="none", color="gray")
ax.plot(0, 0, "k+", ms=12); ax.set_aspect("equal"); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4)
ax.set_title(f"Cursor paths with the {DECODER} decoder"); ax.set_xticks([]); ax.set_yticks([])
plt.show()
ok = [t for t in times if t]
print(f"표적 8개 중 {len(ok)}개 도착, 평균 {np.mean(ok)*0.1:.1f}초 걸림 (곧게 가면 {1/STEP*0.1:.1f}초)")

## 5단계. 보정 데이터를 줄이면?

환자는 매일 BCI를 쓰기 전에 보정을 합니다. 보정은 지루하고 힘들기 때문에 짧을수록 좋습니다. 하지만 너무 줄이면 해독기가 잘 배우지 못합니다. 보정 횟수를 10번부터 400번까지 바꿔 가며, 두 해독기의 정확도를 비교합니다. 우연의 영향을 줄이려고 각 경우를 10번씩 되풀이해 평균을 냅니다.

In [ ]:
calib_sizes = [10, 20, 40, 60, 100, 200, 400]
res = {"population vector": [], "linear decoder": []}
test_dirs, test_counts = collect(400, np.random.default_rng(99))
for n in calib_sizes:
    a_pv, a_lin = [], []
    for rep in range(10):
        r = np.random.default_rng(1000 + rep)
        d, c = collect(n, r)
        a_pv.append(accuracy(decode_population_vector(fit_population_vector(d, c), test_counts), test_dirs))
        a_lin.append(accuracy(decode_linear(fit_linear(d, c), test_counts), test_dirs))
    res["population vector"].append(np.mean(a_pv)); res["linear decoder"].append(np.mean(a_lin))

display(pd.DataFrame({"보정 횟수": calib_sizes,
                      "집단 벡터 정확도": [f"{v:.0%}" for v in res["population vector"]],
                      "선형 해독기 정확도": [f"{v:.0%}" for v in res["linear decoder"]]}))
plt.figure(figsize=(6.5, 4))
for k, v in res.items():
    plt.plot(calib_sizes, np.array(v) * 100, "o-", label=k)
plt.axhline(100/8, color="gray", ls=":", label="random guess (12.5%)")
plt.xscale("log"); plt.xlabel("calibration trials (0.1 s each)"); plt.ylabel("accuracy (%)")
plt.title("More calibration, better decoding"); plt.legend(); plt.show()

**읽는 법**: 보정이 늘수록 두 해독기 모두 좋아집니다. 보정이 10번(1초 분량)뿐이면 정확도가 크게 떨어집니다. 해독기가 뉴런들의 '말투'를 익히기에 너무 짧기 때문입니다. 이 실습에서는 선형 해독기가 대체로 앞섭니다. 모든 뉴런의 스파이크를 한꺼번에 보고 가중치를 맞추기 때문입니다. 대신 집단 벡터는 '어느 뉴런이 어느 방향을 좋아하는지'를 사람이 그대로 읽을 수 있습니다. 더 정확한 해독기와 더 설명하기 쉬운 해독기 사이의 맞바꿈입니다.

## 6단계. 뉴런을 '끊어' 보기

전극이 조금 움직이거나 실이 빠지면 듣던 뉴런의 신호가 끊깁니다. 3단계의 해독기를 **그대로 두고** 뉴런을 몇 개씩 끊어 봅니다. 그다음, 끊어진 상태에서 **다시 보정**한 해독기와 비교합니다.

👉 **바꿔 볼 곳**: `RECALIB_TRIALS`(다시 보정할 때의 횟수)를 바꿔 보세요.

In [ ]:
RECALIB_TRIALS = 100   # ← 이 숫자를 바꿔 보세요 (끊어진 뒤 다시 보정하는 횟수)

lost_list = [0, 5, 10, 20, 30, 40, 45]
rows, curves = [], {k: [] for k in ["PV, no recalibration", "linear, no recalibration", "PV, recalibrated", "linear, recalibrated"]}
for lost in lost_list:
    if lost >= N_NEURONS: break
    alive = np.ones(N_NEURONS)
    alive[np.random.default_rng(7).choice(N_NEURONS, lost, replace=False)] = 0
    td, tc = collect(400, np.random.default_rng(99), alive)
    a1 = accuracy(decode_population_vector(pv, tc), td)          # 옛 해독기 그대로
    a2 = accuracy(decode_linear(W, tc), td)
    d, c = collect(RECALIB_TRIALS, np.random.default_rng(55), alive)   # 다시 보정
    a3 = accuracy(decode_population_vector(fit_population_vector(d, c), tc), td)
    a4 = accuracy(decode_linear(fit_linear(d, c), tc), td)
    for k, v in zip(curves, [a1, a2, a3, a4]): curves[k].append(v)
    rows.append({"끊어진 뉴런 수": lost, "집단 벡터(그대로)": f"{a1:.0%}", "선형(그대로)": f"{a2:.0%}",
                 "집단 벡터(다시 보정)": f"{a3:.0%}", "선형(다시 보정)": f"{a4:.0%}"})
display(pd.DataFrame(rows))

plt.figure(figsize=(6.5, 4))
for k, v in curves.items():
    plt.plot(lost_list[:len(v)], np.array(v) * 100, "o-" if "recal" in k else "s--", label=k)
plt.axhline(100/8, color="gray", ls=":")
plt.xlabel("number of lost neurons"); plt.ylabel("accuracy (%)")
plt.title("Losing neurons, and recalibrating"); plt.legend(fontsize=8); plt.show()

**읽는 법**: 뉴런 몇 개가 끊겨도 정확도는 생각보다 천천히 떨어집니다. 여러 뉴런이 함께 투표하기 때문입니다(6장의 앙상블과 비슷합니다). 그래도 많이 끊기면 옛 해독기는 크게 흔들리고, 남은 뉴런으로 **다시 보정**하면 상당 부분 되살아납니다. 실제 BCI 환자가 전극에 문제가 생겼을 때 해독기를 다시 맞춰 기능을 되찾은 것과 같은 원리입니다.

## 더 해 보기

- 1단계의 `N_NEURONS`를 `4`로 줄이고 모두 다시 실행해 보세요. 종이 실습처럼 뉴런 4개만으로는 얼마나 맞힐 수 있습니까?
- 1단계의 `WINDOW`를 `0.3`으로 늘려 보세요. 더 오래 듣고 판단하면 정확해지지만, 커서는 그만큼 늦게 반응합니다. 정확도와 빠르기 사이의 맞바꿈을 생각해 보세요.

## 생각해 볼 것

1. 5단계에서 보정이 짧을수록 정확도가 떨어졌습니다. 환자가 매일 아침 보정을 해야 한다면, 몇 분까지가 적당할까요? 환자의 피로와 정확도를 함께 생각해 쓰세요.
2. 이 실습의 해독기는 사용자가 '협조해서' 방향을 상상할 때만 작동합니다. 책 4-2절의 '정말 생각을 읽을 수 있을까'와 연결해, 사용자가 협조하지 않으면 어떤 일이 생길지 설명해 보세요.
3. 보정 데이터에는 그 사람만의 뇌 신호 패턴이 담깁니다. 이 데이터는 누구의 것이어야 하고, 회사가 다른 목적으로 쓰지 못하게 하려면 어떤 약속(동의서 항목)이 필요할까요? 책 실습 2의 뇌 데이터 동의서 토론과 연결해 보세요.